# Monitoreo de Fauna mediante YOLOv8

**Notebook limpio para Google Colab.** Este cuaderno organiza la verificación de artefactos, consulta de resultados existentes y generación de resúmenes a partir de CSV ya producidos.

## Reglas de uso
- Ejecutar en Google Colab; Google Drive contiene los datos, pesos y resultados persistentes.
- No entrena modelos automáticamente ni descarga el dataset completo.
- No sobrescribe pesos ni resultados existentes.
- La inferencia nueva es opcional y está desactivada por defecto.
- `val_cis` corresponde a ubicaciones conocidas y `val_trans` a una ubicación no vista; sus resultados deben analizarse por separado.

**Dataset preparado:** 12.099 imágenes de entrenamiento, 1.665 imágenes `val_cis` y 1.533 imágenes `val_trans`; 15 clases de objetos.

## 1. Preparar el entorno de Google Colab

En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** si necesitas ejecutar inferencia. Las celdas de lectura de CSV no requieren GPU.

In [ ]:
# Instalación mínima para inspeccionar resultados y cargar YOLO si se necesita.
# Si Ultralytics ya está instalado, esta celda lo actualizará/instalará según pip.
%pip -q install ultralytics pandas pyyaml matplotlib

In [1]:
from pathlib import Path
import json
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    raise RuntimeError("Este notebook está diseñado para ejecutarse en Google Colab.")

drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/Monitoreo_Fauna_YOLOv8')
DATA_ROOT = PROJECT / '01_datos_yolo'
MODELS_ROOT = PROJECT / '02_modelos'
RESULTS_ROOT = PROJECT / '03_resultados'
REPORTS_ROOT = PROJECT / '04_informes'
LOGS_ROOT = PROJECT / '05_registros'

print('Proyecto existe:', PROJECT.exists())
print('Ruta:', PROJECT)
if not PROJECT.exists():
    raise FileNotFoundError(
        'No se encontró la carpeta del proyecto en Google Drive. '
        'Comprueba el nombre y la ubicación de Monitoreo_Fauna_YOLOv8.'
    )

Mounted at /content/drive
Proyecto existe: True
Ruta: /content/drive/MyDrive/Monitoreo_Fauna_YOLOv8


## 2. Verificar los archivos existentes

Esta sección solo lee el contenido de Google Drive. No modifica ni elimina archivos.

In [ ]:
required_paths = {
    'dataset YOLO': DATA_ROOT,
    'modelo entrenamiento_v1': MODELS_ROOT / 'entrenamiento_v1' / 'weights' / 'best.pt',
    'modelo experimento_416': MODELS_ROOT / 'experimento_416' / 'weights' / 'best.pt',
    'resultados de monitoreo ecológico': RESULTS_ROOT / 'monitoreo_ecologico',
}

inventory = []
for label, path in required_paths.items():
    inventory.append({
        'elemento': label,
        'ruta': str(path),
        'existe': path.exists(),
        'tipo': 'carpeta' if path.is_dir() else ('archivo' if path.is_file() else 'no encontrado'),
        'tamano_MB': round(path.stat().st_size / (1024**2), 2) if path.is_file() else None,
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

## 3. Consultar resultados de evaluación ya guardados

El notebook intentará localizar la tabla de comparación registrada en el proyecto. Si no existe en la ruta esperada, mostrará un aviso en lugar de volver a ejecutar evaluaciones.

In [ ]:
comparison_candidates = [
    RESULTS_ROOT / 'comparacion_val_trans_416.csv',
    RESULTS_ROOT / 'evaluacion' / 'comparacion_val_trans_416.csv',
    PROJECT / 'comparacion_val_trans_416.csv',
]
comparison_file = next((p for p in comparison_candidates if p.exists()), None)

if comparison_file is None:
    print('No se encontró comparacion_val_trans_416.csv en las rutas habituales.')
    print('Busca el CSV en 03_resultados y actualiza comparison_candidates si está en otra subcarpeta.')
else:
    print('Archivo:', comparison_file)
    comparison_df = pd.read_csv(comparison_file)
    display(comparison_df.head(20))

## 4. Revisar los reportes del módulo de monitoreo ecológico

Se leen los CSV existentes generados por la inferencia anterior. Esta sección **no vuelve a ejecutar YOLO**.

In [ ]:
ECO_ROOT = RESULTS_ROOT / 'monitoreo_ecologico'
report_names = [
    'detecciones_val_trans.csv',
    'resumen_por_imagen.csv',
    'actividad_diaria_por_especie.csv',
    'actividad_por_franja_horaria.csv',
    'resumen_general_por_especie.csv',
]

report_inventory = []
for name in report_names:
    path = ECO_ROOT / name
    row = {'archivo': name, 'existe': path.exists(), 'ruta': str(path)}
    if path.exists():
        try:
            df = pd.read_csv(path)
            row['filas'] = len(df)
            row['columnas'] = len(df.columns)
        except Exception as exc:
            row['error_lectura'] = str(exc)
    report_inventory.append(row)

display(pd.DataFrame(report_inventory))

In [ ]:
summary_file = ECO_ROOT / 'resumen_general_por_especie.csv'
if summary_file.exists():
    species_summary = pd.read_csv(summary_file)
    display(species_summary)
else:
    print('No existe resumen_general_por_especie.csv. No se generará de nuevo automáticamente.')

## 5. Opcional: ejecutar inferencia sobre `val_trans`

La inferencia está desactivada por defecto para evitar costes de GPU y no repetir trabajo. Actívala solo si necesitas generar nuevas predicciones y has confirmado que el archivo de salida no existe o quieres una ejecución nueva.

Parámetros documentados en el experimento previo: `imgsz=416`, `conf=0.25`, `iou=0.70`, modelo `entrenamiento_v1/best.pt`. La inferencia nueva crea una carpeta de salida separada y no modifica los pesos.

In [ ]:
RUN_NEW_INFERENCE = False  # Cambiar a True solo si realmente se necesita una inferencia nueva.

if RUN_NEW_INFERENCE:
    from ultralytics import YOLO
    import torch

    model_path = MODELS_ROOT / 'entrenamiento_v1' / 'weights' / 'best.pt'
    val_images = DATA_ROOT / 'images' / 'val_trans'
    output_project = RESULTS_ROOT / 'monitoreo_ecologico_reejecucion'

    if not model_path.exists():
        raise FileNotFoundError(f'No se encuentra el modelo: {model_path}')
    if not val_images.exists():
        raise FileNotFoundError(f'No se encuentra val_trans: {val_images}')
    if output_project.exists():
        raise FileExistsError(
            f'La carpeta de salida ya existe: {output_project}. '
            'Cambia el nombre de la carpeta de salida para no sobrescribir resultados.'
        )

    device = 0 if torch.cuda.is_available() else 'cpu'
    model = YOLO(str(model_path))
    model.predict(
        source=str(val_images),
        imgsz=416,
        conf=0.25,
        iou=0.70,
        device=device,
        save=True,
        project=str(output_project),
        name='predicciones',
        exist_ok=False,
    )
else:
    print('Inferencia desactivada. Se conservan los resultados existentes.')

## 6. Interpretación y limitaciones

- Las métricas de `val_cis` y `val_trans` no son intercambiables: `val_trans` representa una ubicación no vista durante el entrenamiento.
- Las detecciones son resultados del modelo, no un conteo definitivo de abundancia poblacional.
- Los análisis exploratorios de tamaño, oclusión y Grad-CAM deben presentarse con sus limitaciones y tamaño muestral.
- Antes de presentar cifras, verificar la ruta, las columnas y el contexto de cada CSV.

## 7. Información del proyecto

- Plataforma de ejecución: Google Colab.
- Almacenamiento persistente: Google Drive.
- Detector: Ultralytics YOLOv8.
- Clases: 15 categorías de animales/objetos preparadas en formato YOLO.
- Los datos originales y los pesos no se incluyen en GitHub; se mantienen en Google Drive y sujetos a las condiciones de licencia de la fuente.